In [2]:
# Импорты
import os
from pathlib import Path
from typing import List, Dict
import requests
from io import BytesIO

import pdfplumber 
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

import numpy as np
import json

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

In [4]:
PROJECT_ROOT = Path.cwd().parent
data_processed_dir = PROJECT_ROOT / "data" / "processed"
chunks_dir = PROJECT_ROOT / "data" / "chunks"
embeddings_dir = PROJECT_ROOT / "embeddings"

In [ ]:
embeddings_dir.mkdir(parents=True, exist_ok=True)

# Путь к файлам
metadata_path = embeddings_dir / "metadata.json"
embeddings_path = embeddings_dir / "embeddings.npy"

# Создание пустого metadata.json
if not metadata_path.exists():
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump([], f)

# Создание пустого embeddings.npy
if not embeddings_path.exists():
    np.save(embeddings_path, np.empty((0,)))


In [3]:
# Параметры для разбиения текста
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

#  Ссылка на книгу 
PDF_URL = "https://medialex.brsu.by/NLP-BOOK/%C2%AB%D0%93%D0%BB%D1%83%D0%B1%D0%BE%D0%BA%D0%BE%D0%B5%20%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5%C2%BB,%20%D0%AF%D0%BD%20%D0%93%D1%83%D0%B4%D1%84%D0%B5%D0%BB%D0%BB%D0%BE%D1%83.pdf"
BOOK_NAME = "deep_learning_goodfellow.pdf"


In [5]:
def pdf_to_text_from_url(pdf_url: str) -> str:
    """
    Скачивает PDF по ссылке и возвращает весь текст как строку.
    """
    print("Скачиваю PDF из интернета")
    response = requests.get(pdf_url)
    pdf_file = BytesIO(response.content)

    print("Извлекаю текст из PDF")
    text_pages = []
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text_pages.append(page.extract_text() or "")  # на случай пустой страницы

    full_text = "\n".join(text_pages)
    print("Текст извлечён, страниц:", len(text_pages))
    return full_text

In [ ]:
raw_text = pdf_to_text_from_url(PDF_URL)

# Сохраняем текстовую версию книги
processed_path = data_processed_dir / "deep_learning_goodfellow.txt"
with open(processed_path, "w", encoding="utf-8") as f:
    f.write(raw_text)

print("Текст книги сохранён в:", processed_path)

Скачиваю PDF из интернета
Извлекаю текст из PDF
Текст извлечён, страниц: 653
Текст книги сохранён в: c:\КИНО\RAG-Project\data\processed\deep_learning_goodfellow.txt


In [5]:
# Загружаем текст из сохранённого файла
processed_path = data_processed_dir / "deep_learning_goodfellow.txt"
with open(processed_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [6]:
# Разбиение текста на чанки 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(raw_text)
print(f"Количество чанков: {len(chunks)}")

Количество чанков: 4614


In [7]:
# Сохраняем чанки и метаданные
metadata = []
for i, chunk in enumerate(chunks):
    chunk_filename = f"{BOOK_NAME}_chunk_{i}.txt"
    chunk_path = chunks_dir / chunk_filename
    with open(chunk_path, "w", encoding="utf-8") as f:
        f.write(chunk)
    metadata.append({
        "chunk_id": i,
        "source": BOOK_NAME,
        "chunk_file": str(chunk_filename)
    })

In [8]:
# Генерация эмбеддингов 
model = SentenceTransformer("multi-qa-mpnet-base-dot-v1")
texts = [chunk for chunk in chunks]

embeddings = model.encode(texts, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)


c:\КИНО\RAG-Project\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\natas\.cache\huggingface\hub\models--sentence-transformers--multi-qa-mpnet-base-dot-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back t

Embeddings shape: (4614, 768)


In [12]:
# Сохранение
embeddings_path = embeddings_dir / "embeddings.npy"
metadata_path = embeddings_dir / "metadata.json"

# Сохраняем эмбеддинги
np.save(embeddings_path, embeddings)

# Сохраняем метаданные
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Embeddings и метаданные сохранены в:")
print("-", embeddings_path)
print("-", metadata_path)

Embeddings и метаданные сохранены в:
- c:\КИНО\RAG-Project\embeddings\embeddings.npy
- c:\КИНО\RAG-Project\embeddings\metadata.json
